# Ponto de Controle
Este notebook valida e escreve dados transformados no Google Sheets.

In [ ]:
import os
import pandas as pd
import datetime as dt
from extract import read_df
from treat.utils.datas import normalize_date_to_str_DD_M_YYYY
from treat.utils.write_dataframe_to_sheet import write_dataframe_to_sheet
from treat.utils.renomeacoes import renomear_colunas_origem_para_modelo as rename
from treat.utils.normalize import normalize_vehicle
from treat.utils.datas import concat_period
from treat.utils.campos_calculados import make_id_ponto_de_controle

In [ ]:
# Flags de execução
"""
Célula  – Imports & parâmetros globais

Define:
- Módulos padrão e helpers do projeto
- Flags de execução e IDs de planilhas via env var
- Constantes de aba, cabeçalho e filtro de data
- Lista de colunas de destino
"""
DRY_RUN = True

# IDs das planilhas via variáveis de ambiente
os.environ["ORIGIN_SHEET_ID"] = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
ORIGIN_SHEET_ID = os.getenv("ORIGIN_SHEET_ID")
os.environ["DEST_SHEET_ID"] = "1DpH5tu4KJKqbA6ueFtf1s1FueBkR4-EtPf5xHyXx8zw"
DEST_SHEET_ID   = os.getenv("DEST_SHEET_ID")

# Garantia de que foram definidas
assert ORIGIN_SHEET_ID is not None and DEST_SHEET_ID is not None, \
    "Defina as variáveis de ambiente ORIGIN_SHEET_ID e DEST_SHEET_ID"


In [ ]:
# Constantes de aba & cabeçalho: nomes centralizados em um só lugar
ORIGIN_TAB    = "modeloGeral"
DEST_TAB      = "IMPULSIONAMENTOS 2025"
HEAD_ROW_DEST = 4  # zero-based (header na linha 5)



In [ ]:
# Filtro temporal & Data mínima – usado no filtro posterior
MIN_DATE = dt.date(2025, 6, 1)


In [ ]:
# Lista DEST_COLUMNS: declara lista com 11 colunas, ordem exata exigida
DEST_COLUMNS = [
    "Data",
    "Campanha",
    "Veiculo",
    "Link conteúdos impulsionados",
    "Período",
    "Agência",
    "Editoria",
    "Objetivo (aumentar seguidores, melhorar engajamento, etc)",
    "Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)",
    "Status",
    "Resultado",
]
assert len(DEST_COLUMNS) == 11, f"DEST_COLUMNS deve ter 11 colunas, mas tem {len(DEST_COLUMNS)}"
    

In [ ]:
# Leitura da aba de origem – deve executar sem exceção se ORIGIN_SHEET_ID estiver definido
"""
Célula 2 – Leitura + filtro temporal da aba modeloGeral
- Lê df_origin com read_df()
- Converte coluna date para date_dt
- Filtra linhas >= MIN_DATE
- Garante colunas críticas e prepara df_origin
"""
df_origin = read_df(
    sheet_id=ORIGIN_SHEET_ID,
    tab=ORIGIN_TAB,
    header_row=0,
)

In [ ]:
# Sanitizar tipos de data – converte coluna 'date' para datetime.date e força dtype str para preservar zeros
df_origin['date'] = df_origin['date'].astype(str)
df_origin['date_dt'] = pd.to_datetime(df_origin['date'], errors='coerce').dt.date

# Preview das datas para validação
if DRY_RUN:
    display(df_origin[['date', 'date_dt']].head())
    invalidados = df_origin['date_dt'].isna().sum()
    print(f"Valores inválidos ou não parseados: {invalidados}")


In [ ]:
# Quick-preview em DRY_RUN – exibe head e contagem somente em Dry Run
if DRY_RUN:
    display(df_origin.head())
    print(f"Total de linhas em df_origin: {len(df_origin)}")


In [ ]:
# Assert de colunas críticas – garante que df_origin tenha todas as colunas necessárias
required_columns = ["date", "Campanha", "Veiculo", "URL_do_Anuncio", "objective"]
missing_cols = [col for col in required_columns if col not in df_origin.columns]
if missing_cols:
    raise RuntimeError(f"Colunas críticas ausentes em df_origin: {missing_cols}")


In [ ]:
# Clean-up da coluna auxiliar – remover 'date_dt' apenas após aplicar o filtro de data mínima (útil para debug)
if not DRY_RUN:
    df_origin.drop(columns=["date_dt"], inplace=True)


In [ ]:
# Clonar DataFrame para não mutar a leitura crua
df = df_origin.copy()


In [ ]:
# Normalizar coluna Data e dropar coluna date após criar Data para evitar conflito de nomes
df["Data"] = df["date"].apply(normalize_date_to_str_DD_M_YYYY)
df.drop(columns=["date"], inplace=True)

# Debug: mostrar os primeiros valores de 'Data' e conferir dtype
print("Preview 'Data':")
print(df["Data"].head())
print(f"Tipo de coluna Data: {df['Data'].dtype}, total de linhas: {len(df)}")


In [ ]:
# Normalizar Veiculo – aplicação de normalize_vehicle
df["Veiculo"] = df["Veiculo"].apply(normalize_vehicle)


In [ ]:
# Gerar coluna Período com concat_period e dropar colunas auxiliares
df["Período"] = df.apply(lambda r: concat_period(r.get("start"), r.get("end")), axis=1)
df.drop(columns=["start", "end"], inplace=True)

# Debug: conferir se a geração de 'Período' funcionou
print("Preview 'Período':")
print(df["Período"].head().tolist())
non_empty = df["Período"].astype(bool).sum()
print(f"Linhas com período não vazio: {non_empty} de {len(df)}")


In [ ]:
# Mapear colunas diretas
df["Campanha"] = df["Campanha"]
df["Link conteúdos impulsionados"] = df["URL_do_Anuncio"]
df["Objetivo (aumentar seguidores, melhorar engajamento, etc)"] = df["objective"]

# Debug prints para verificar mapeamento
print("Preview 'Campanha':", df["Campanha"].head().tolist())
print("Preview 'Link conteúdos impulsionados':", df["Link conteúdos impulsionados"].head().tolist())
print("Preview 'Objetivo (aumentar seguidores, melhorar engajamento, etc)':",
      df["Objetivo (aumentar seguidores, melhorar engajamento, etc)"].head().tolist())

# Asserts para garantir a presença das colunas
assert "Campanha" in df.columns, "Coluna 'Campanha' não encontrada"
assert "Link conteúdos impulsionados" in df.columns, "Coluna 'Link conteúdos impulsionados' não encontrada"
assert "Objetivo (aumentar seguidores, melhorar engajamento, etc)" in df.columns, \
       "Coluna 'Objetivo (aumentar seguidores, melhorar engajamento, etc)' não encontrada"


In [ ]:
# Colunas constantes / vazias
df["Agência"] = "De Brito"
df["Editoria"] = df["Campanha"]
df["Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)"] = ""
df["Status"] = ""
df["Resultado"] = ""

# Debug prints para verificar preenchimento
print("Preview 'Agência':", df["Agência"].head().tolist())
print("Preview 'Editoria':", df["Editoria"].head().tolist())
print("Valores únicos em 'Agência':", df["Agência"].unique())
print("Contagem não vazia em 'Meta (número)':",
      df["Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)"].astype(bool).sum())
print("Contagem não vazia em 'Status':",
      df["Status"].astype(bool).sum())
print("Contagem não vazia em 'Resultado':",
      df["Resultado"].astype(bool).sum())

# Asserts para garantir colunas e conteúdo esperado
assert "Agência" in df.columns and df["Agência"].eq("De Brito").all(), \
    "Erro em 'Agência': valores diferentes de 'De Brito' ou coluna ausente"
assert "Editoria" in df.columns, "Coluna 'Editoria' ausente"
assert all(df["Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)"] == ""), \
    "'Meta (número)' deve ser completamente vazia"
assert all(df["Status"] == ""), "'Status' deve ser completamente vazio"
assert all(df["Resultado"] == ""), "'Resultado' deve ser completamente vazio"


In [ ]:
# Reordenar / reindexar com DEST_COLUMNS e preencher vazios
df_transf = df.reindex(columns=DEST_COLUMNS, fill_value="")

# Debug prints e asserts para verificar ordem e conteúdo das colunas
print("Colunas em df_transf:", df_transf.columns.tolist())
assert df_transf.columns.tolist() == DEST_COLUMNS, (
    f"Colunas fora de ordem ou faltando: {df_transf.columns.tolist()}"
)

# Mostrar as primeiras linhas para confirmação visual
display(df_transf.head())
print(f"Total de linhas em df_transf: {len(df_transf)}")


In [ ]:
# 3.7.1 – Dropar colunas auxiliares 'start' e 'end' se ainda existirem
aux_cols = [c for c in ["start", "end"] if c in df_transf.columns]
if aux_cols:
    df_transf.drop(columns=aux_cols, inplace=True)

# Debug: confirmar que as colunas auxiliares foram removidas
print("Colunas após remoção de 'start' e 'end':", df_transf.columns.tolist())


In [ ]:
# ---------------------------------------------------------------------------
# Leitura da aba de destino  “IMPULSIONAMENTOS 2025”
# ---------------------------------------------------------------------------
from extract import read_df   # helper já usado antes

# 1 • Lê a aba usando cabeçalho real na linha 5 (zero-based 4)
df_dest = read_df(
    sheet_id=DEST_SHEET_ID,
    tab=DEST_TAB,
    header_row=HEAD_ROW_DEST - 1,   # HEAD_ROW_DEST = 5  → 4
)

# 2 • Remove eventuais colunas duplicadas/vazias geradas pelo Sheets
if df_dest.columns.duplicated().any() or "" in df_dest.columns:
    print("⚠️  Cabeçalho continha rótulos duplicados ou vazios – normalizando…")
    df_dest = df_dest.loc[:, ~df_dest.columns.duplicated()]           # remove duplicadas
    df_dest.rename(columns=lambda c: c if c else "_blank", inplace=True)

# 3 • Mantém apenas as 11 colunas oficiais e preserva a ordem
df_dest = df_dest.reindex(columns=DEST_COLUMNS, fill_value="")

# 4 • Descarta linhas totalmente vazias (apenas bordas/formatação)
df_dest = df_dest.replace("", pd.NA).dropna(how="all").reset_index(drop=True)

# 5 • Valida e mostra prévia
assert df_dest.columns.tolist() == DEST_COLUMNS, \
    f"Colunas inesperadas: {df_dest.columns.tolist()}"
print(f"Linhas válidas em df_dest: {len(df_dest)}")
display(df_dest.head())

# Agora df_dest está pronto para gerar __ID__, deduplicar e gravar


⚠️  Header bruto contém duplicados ou vazios. Normalizando…
Colunas lidas: ['Data', 'Campanha', 'Veiculo', 'Link conteúdos impulsionados', 'Período', 'Agência', 'Editoria', 'Objetivo (aumentar seguidores, melhorar engajamento, etc)', 'Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)', 'Status', 'Resultado']


,Data,Campanha,Veiculo,Link conteúdos impulsionados,Período,Agência,Editoria,"Objetivo (aumentar seguidores, melhorar engajamento, etc)","Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)",Status,Resultado
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,,,,,,,


Total de linhas em df_dest: 833
Linhas com pelo menos 1 valor: 0


,Data,Campanha,Veiculo,Link conteúdos impulsionados,Período,Agência,Editoria,"Objetivo (aumentar seguidores, melhorar engajamento, etc)","Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)",Status,Resultado


=== Primeiras 10 linhas brutas ===


,Data,Campanha,Veiculo,Link conteúdos impulsionados,Período,Agência,Editoria,"Objetivo (aumentar seguidores, melhorar engajamento, etc)","Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)",Status,Resultado
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,,,,,,,
5,,,,,,,,,,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


=== Quantidade de valores não vazios por coluna ===
Data                                                                      0
Campanha                                                                  0
Veiculo                                                                   0
Link conteúdos impulsionados                                              0
Período                                                                   0
Agência                                                                   0
Editoria                                                                  0
Objetivo (aumentar seguidores, melhorar engajamento, etc)                 0
Meta (número) (quantos seguidores, compartilhamentos, etc. previstos?)    0
Status                                                                    0
Resultado                                                                 0
dtype: int64


In [ ]:
# 4.1 Debug avançado: inspecionar o “raw” do df_dest sem header para descobrir onde está o cabeçalho real
raw_dest = read_df(
    sheet_id=DEST_SHEET_ID,
    tab=DEST_TAB,
    header_row=0,     # força ler tudo como dados sem extrair header
)
print("=== Primeiras 8 linhas brutas de df_dest ===")
display(raw_dest.head(8))

# Tentar localizar dinamicamente qual linha contém o nome "Data"
detected = None
for idx, row in raw_dest.iterrows():
    if "Data" in row.values:
        detected = idx
        break

print(f"Header detectado na linha (zero‐based): {detected}")

# Re-ler df_dest usando a linha detectada como header_row
if detected is not None:
    df_dest = read_df(
        sheet_id=DEST_SHEET_ID,
        tab=DEST_TAB,
        header_row=detected,
    )
    print(f"✅ df_dest recarregado com header_row={detected}")
    display(df_dest.head())
    print("Colunas finais em df_dest:", df_dest.columns.tolist())
else:
    raise RuntimeError("Não foi possível detectar automaticamente a linha de cabeçalho em df_dest")
